# 05 — Inference-time Preprocessing

Per-noise word-level normalization pipelines applied to the noisy CoNLL test sets,
then evaluated on 4 frozen baseline models. Preprocessors preserve word count so
BIO labels remain aligned. See `docs/specs/2026-04-17-noise-mitigation-design.md`.

In [1]:
import sys, os, glob, json
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, load_from_disk, Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    BertForTokenClassification,
    GPT2ForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    T5Config,
)
from train import (
    ByT5ForTokenClassification,
    tokenize_and_align_labels,
    make_compute_metrics,
)
from preprocess import (
    ocr_pipeline,
    asr_pipeline,
    social_pipeline,
    build_truecase_dict,
    load_truecase_dict,
    save_truecase_dict,
)

In [2]:
if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print(f"Device: {DEVICE}")

label_names = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']
id2label = {i: l for i, l in enumerate(label_names)}
label2id = {l: i for i, l in enumerate(label_names)}
num_labels = len(label_names)

ROOT = os.path.dirname(os.getcwd())
RESULTS_DIR = os.path.join(ROOT, "results")
TABLES_DIR = os.path.join(RESULTS_DIR, "tables")
CHECKPOINTS_DIR = os.path.join(RESULTS_DIR, "checkpoints")
NOISY_DIR = os.path.join(ROOT, "data", "noisy")
PREPROC_DIR = os.path.join(ROOT, "data", "preprocessed")
os.makedirs(PREPROC_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)

Device: mps


In [3]:
def tokenize_and_align_labels_byt5(examples, tokenizer, max_length=256):
    all_input_ids, all_attention_masks, all_labels = [], [], []
    for tokens, ner_tags in zip(examples["tokens"], examples["ner_tags"]):
        input_ids = [0]
        label_ids = [-100]
        for word, label in zip(tokens, ner_tags):
            word_ids = tokenizer(word, add_special_tokens=False)["input_ids"]
            input_ids += word_ids
            label_ids += [label] + [-100] * (len(word_ids) - 1)
        input_ids = input_ids[:max_length]
        label_ids = label_ids[:max_length]
        pad_len = max_length - len(input_ids)
        all_input_ids.append(input_ids + [0] * pad_len)
        all_attention_masks.append([1]*len(input_ids) + [0]*pad_len)
        all_labels.append(label_ids + [-100]*pad_len)
    return {"input_ids": all_input_ids, "attention_mask": all_attention_masks, "labels": all_labels}

## Build truecase dict from CoNLL train

In [4]:
clean = load_dataset("conll2003", trust_remote_code=True)
truecase = build_truecase_dict(clean["train"])
truecase_path = os.path.join(PREPROC_DIR, "truecase_dict.json")
save_truecase_dict(truecase, truecase_path)
load_truecase_dict(truecase_path)  # hot-reload into asr_truecase module default
print(f"Truecase dict: {len(truecase)} entries  (saved: {truecase_path})")
print("Sample:", {k: truecase[k] for k in list(truecase.keys())[:8]})

Truecase dict: 21009 entries  (saved: /Users/narly/Code/Study/S26/NLP/S26-NLP-Tokenization-for-Noisy-Texts/data/preprocessed/truecase_dict.json)
Sample: {'eu': 'EU', 'rejects': 'rejects', 'german': 'German', 'call': 'call', 'to': 'to', 'boycott': 'boycott', 'british': 'British', 'lamb': 'lamb'}


## Apply preprocessors to noisy test sets

In [5]:
PIPELINES = {"ocr": ocr_pipeline, "asr": asr_pipeline, "social": social_pipeline}

for noise_type, pipe in PIPELINES.items():
    src_path = os.path.join(NOISY_DIR, noise_type)
    dst_path = os.path.join(PREPROC_DIR, noise_type)
    print(f"\n=== {noise_type}: {src_path} -> {dst_path} ===")
    noisy_ds = load_from_disk(src_path)

    def _apply(example):
        orig = example["tokens"]
        out = pipe(orig)
        assert len(out) == len(orig), f"len mismatch: {len(orig)} -> {len(out)}"
        example["tokens"] = out
        return example

    pre_ds = DatasetDict({split: ds.map(_apply) for split, ds in noisy_ds.items()})
    pre_ds.save_to_disk(dst_path)

    # Sample pre/post
    i = 0
    print("  noisy :", noisy_ds["test"][i]["tokens"][:18])
    print("  preproc:", pre_ds["test"][i]["tokens"][:18])


=== ocr: /Users/narly/Code/Study/S26/NLP/S26-NLP-Tokenization-for-Noisy-Texts/data/noisy/ocr -> /Users/narly/Code/Study/S26/NLP/S26-NLP-Tokenization-for-Noisy-Texts/data/preprocessed/ocr ===


Saving the dataset (0/1 shards):   0%|          | 0/14041 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3250 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3453 [00:00<?, ? examples/s]

  noisy : ['5OCC3R', '-', 'JAPAN', 'GETLUCKY', 'WIN', ',', 'CHINA', 'IN', 'SURPRISEDEFEAT', '.']
  preproc: ['sOCCeR', '-', 'JAPAN', 'GETLUCKY', 'WIN', ',', 'CHINA', 'IN', 'SURPRISEDEFEAT', '.']

=== asr: /Users/narly/Code/Study/S26/NLP/S26-NLP-Tokenization-for-Noisy-Texts/data/noisy/asr -> /Users/narly/Code/Study/S26/NLP/S26-NLP-Tokenization-for-Noisy-Texts/data/preprocessed/asr ===


...

  noisy : ['soccer', 'japan', 'get', 'lucky', 'win', 'china', 'in', 'surprise', 'defeat']
  preproc: ['SOCCER', 'Japan', 'get', 'lucky', 'win', 'China', 'in', 'surprise', 'defeat']

=== social: /Users/narly/Code/Study/S26/NLP/S26-NLP-Tokenization-for-Noisy-Texts/data/noisy/social -> /Users/narly/Code/Study/S26/NLP/S26-NLP-Tokenization-for-Noisy-Texts/data/preprocessed/social ===


Saving the dataset (0/1 shards):   0%|          | 0/14041 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3250 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3453 [00:00<?, ? examples/s]

  noisy : ['SlCCER', '-', 'JAoAN', 'fET', 'LUCKY', 'WIN', ',', 'dHINA', 'IN', 'SURPRISE', 'DEFEAT', '.']
  preproc: ['soccer', '-', 'joan', 'get', 'LUCKY', 'WIN', ',', 'china', 'IN', 'SURPRISE', 'DEFEAT', '.']


## Load checkpoints (4 models)

In [6]:
# Model loading & per-model evaluation was performed out-of-notebook via
# `scripts/eval_preprocess_model.py <model_name>` to keep MPS/process memory
# isolated. Each subprocess writes a JSON record to
# results/tables/preprocess_partials/<model>.json with per-noise metrics.
# The next cell aggregates these JSONs into `results_df`.

print("Evaluation dispatched via subprocess script; see scripts/eval_preprocess_model.py")


Evaluation dispatched via subprocess script; see scripts/eval_preprocess_model.py


## Evaluate on preprocessed noisy test sets

In [7]:
# Eval was executed via scripts/eval_preprocess_model.py in isolated subprocesses
# to avoid MPS memory accumulation across models. See commit for rationale.
# Here we aggregate the per-model partial JSONs produced by those runs.

PARTIALS_DIR = os.path.join(TABLES_DIR, "preprocess_partials")

MODEL_NAMES = ["bert-base-uncased", "bert-base-cased", "gpt2", "google/byt5-small"]

all_records = []
for name in MODEL_NAMES:
    safe = name.replace("/", "__")
    path = os.path.join(PARTIALS_DIR, f"{safe}.json")
    with open(path, "r", encoding="utf-8") as f:
        recs = json.load(f)
    all_records.extend(recs)

results_df = pd.DataFrame(all_records)
print("Loaded partials for", MODEL_NAMES)
results_df


Loaded partials for ['bert-base-uncased', 'bert-base-cased', 'gpt2', 'google/byt5-small']


,model,noise,f1,precision,recall
0,bert-base-uncased,ocr,0.7831,0.7588,0.8091
1,bert-base-uncased,asr,0.8297,0.8274,0.8320
2,bert-base-uncased,social,0.7318,0.6763,0.7973
3,bert-base-cased,ocr,0.7996,0.8106,0.7889
4,bert-base-cased,asr,0.6746,0.7073,0.6448
5,bert-base-cased,social,0.7396,0.7377,0.7415
6,gpt2,ocr,0.5978,0.6001,0.5956
7,gpt2,asr,0.4620,0.4830,0.4428
8,gpt2,social,0.5436,0.5304,0.5575
9,google/byt5-small,ocr,0.7847,0.7884,0.7810


## Delta vs noisy baseline + clean baseline

In [8]:
# Clean baselines: baseline_results.csv (3 models) + bert_cased_clean_results.csv
base_clean = pd.read_csv(os.path.join(TABLES_DIR, "baseline_results.csv"))[["model", "f1_clean"]]
cased_clean = pd.read_csv(os.path.join(TABLES_DIR, "bert_cased_clean_results.csv"))[["model", "f1_clean"]]
clean_df = pd.concat([base_clean, cased_clean], ignore_index=True)
clean_df.columns = ["model", "f1_clean"]

# Noisy baselines: from notebook 03 (3 models) and notebook 04 (bert-cased)
noisy_eval = pd.read_csv(os.path.join(TABLES_DIR, "noisy_evaluation.csv"))[["model", "noise", "f1"]]
noisy_eval.columns = ["model", "noise", "f1_noisy"]
cased_noisy = pd.read_csv(os.path.join(TABLES_DIR, "bert_cased_noisy_results.csv"))[["model", "noise", "f1"]]
cased_noisy.columns = ["model", "noise", "f1_noisy"]
noisy_df = pd.concat([noisy_eval, cased_noisy], ignore_index=True)

merged = results_df.rename(columns={"f1": "f1_preprocess"}).merge(clean_df, on="model", how="left")
merged = merged.merge(noisy_df, on=["model", "noise"], how="left")
merged["f1_uplift"] = (merged["f1_preprocess"] - merged["f1_noisy"]).round(4)
merged["f1_recovered_pct"] = ((merged["f1_uplift"] / (merged["f1_clean"] - merged["f1_noisy"])) * 100).round(2)

# Re-order columns
merged = merged[[
    "model", "noise", "f1_preprocess", "precision", "recall",
    "f1_clean", "f1_noisy", "f1_uplift", "f1_recovered_pct",
]]
merged

,model,noise,f1_preprocess,precision,recall,f1_clean,f1_noisy,f1_uplift,f1_recovered_pct
0,bert-base-uncased,ocr,0.7831,0.7588,0.8091,0.890836,0.6632,0.1199,52.67
1,bert-base-uncased,asr,0.8297,0.8274,0.8320,0.890836,0.8255,0.0042,6.43
2,bert-base-uncased,social,0.7318,0.6763,0.7973,0.890836,0.6174,0.1144,41.84
3,bert-base-cased,ocr,0.7996,0.8106,0.7889,0.896515,0.6998,0.0998,50.73
4,bert-base-cased,asr,0.6746,0.7073,0.6448,0.896515,0.2134,0.4612,67.51
5,bert-base-cased,social,0.7396,0.7377,0.7415,0.896515,0.7096,0.0300,16.05
6,gpt2,ocr,0.5978,0.6001,0.5956,0.687350,0.5433,0.0545,37.83
7,gpt2,asr,0.4620,0.4830,0.4428,0.687350,0.0322,0.4298,65.60
8,gpt2,social,0.5436,0.5304,0.5575,0.687350,0.5296,0.0140,8.87
9,google/byt5-small,ocr,0.7847,0.7884,0.7810,0.862590,0.7018,0.0829,51.56


In [9]:
out_path = os.path.join(TABLES_DIR, "preprocess_results.csv")
merged.to_csv(out_path, index=False)
print(f"Saved {out_path}")

Saved /Users/narly/Code/Study/S26/NLP/S26-NLP-Tokenization-for-Noisy-Texts/results/tables/preprocess_results.csv


In [10]:
pivot = merged.pivot(index="model", columns="noise", values="f1_uplift")
print("\nF1 uplift per model x noise (preprocessed - noisy baseline):")
print(pivot.to_string())

pivot_rec = merged.pivot(index="model", columns="noise", values="f1_recovered_pct")
print("\nRecovered %% of clean-vs-noisy gap:")
print(pivot_rec.to_string())


F1 uplift per model x noise (preprocessed - noisy baseline):
noise                 asr     ocr  social
model                                    
bert-base-cased    0.4612  0.0998  0.0300
bert-base-uncased  0.0042  0.1199  0.1144
google/byt5-small  0.1996  0.0829  0.0273
gpt2               0.4298  0.0545  0.0140

Recovered %% of clean-vs-noisy gap:
noise                asr    ocr  social
model                                  
bert-base-cased    67.51  50.73   16.05
bert-base-uncased   6.43  52.67   41.84
google/byt5-small  42.28  51.56   17.64
gpt2               65.60  37.83    8.87
